In [3]:
import mpmath as mp
from dataclasses import dataclass

@dataclass
class ShootResult:
    energy: mp.mpf
    parity: str
    nodes_halfline: int
    psi_rmax: mp.mpf
    success: bool
    message: str


class AnharmonicOscillator1DMP:
    """
    Arbitrary-precision solver for
        H = 1/2 p^2 + 1/2 x^2 + lam4/4! x^4 + lam6/6! x^6
    using parity-reduced shooting on x >= 0.

    Solves:
        psi'' = [x^2 + 2 lam4/4! x^4 + 2 lam6/6! x^6 - 2E] psi
    """

    def __init__(
        self,
        lam4=0,
        lam6=0,
        r0=0,
        rmax=5,
        dps=50,
        n_count_grid=2000,
    ):
        mp.mp.dps = dps

        self.lam4 = mp.mpf(lam4)
        self.lam6 = mp.mpf(lam6)
        self.r0 = mp.mpf(r0)
        self.rmax = mp.mpf(rmax)
        self.n_count_grid = int(n_count_grid)

        self.fact4 = mp.mpf(24)
        self.fact6 = mp.mpf(720)

    def set_precision(self, dps):
        mp.mp.dps = int(dps)

    def V_eff_twice(self, x):
        return (
            x**2
            + 2 * self.lam4 * x**4 / self.fact4
            + 2 * self.lam6 * x**6 / self.fact6
        )

    def ode_rhs(self, x, y, E):
        psi, dpsi = y
        ddpsi = (self.V_eff_twice(x) - 2 * E) * psi
        return [dpsi, ddpsi]

    def initial_conditions(self, parity):
        parity = parity.lower()
        if parity == "even":
            return [mp.mpf(1), mp.mpf(0)]
        elif parity == "odd":
            return [mp.mpf(0), mp.mpf(1)]
        else:
            raise ValueError("parity must be 'even' or 'odd'")

    def solve_wavefunction(self, E, parity):
        """
        Build an arbitrary-precision ODE solution with mpmath.
        """
        E = mp.mpf(E)
        y0 = self.initial_conditions(parity)

        try:
            f = lambda x, y: self.ode_rhs(x, y, E)
            sol = mp.odefun(f, self.r0, y0)
            return sol
        except Exception as exc:
            return exc

    def count_nodes_halfline(self, sol, parity):
        """
        Count sign changes in psi on (0, rmax].
        """
        parity = parity.lower()
        n = self.n_count_grid
        dx = (self.rmax - self.r0) / n

        xs = [self.r0 + i * dx for i in range(n + 1)]
        vals = [sol(x)[0] for x in xs]

        if parity == "odd":
            vals = vals[1:]

        # robust sign extraction
        eps = mp.mpf(10) ** (-(mp.mp.dps // 2))
        signs = []
        for v in vals:
            if abs(v) < eps:
                signs.append(0)
            elif v > 0:
                signs.append(1)
            else:
                signs.append(-1)

        # fill zero signs by nearest previous nonzero where possible
        cleaned = []
        prev = None
        for s in signs:
            if s == 0:
                cleaned.append(prev if prev is not None else 0)
            else:
                cleaned.append(s)
                prev = s

        # if initial values stayed zero-like, fill forward
        for i in range(len(cleaned) - 2, -1, -1):
            if cleaned[i] == 0:
                cleaned[i] = cleaned[i + 1]

        nodes = 0
        for a, b in zip(cleaned[:-1], cleaned[1:]):
            if a != 0 and b != 0 and a != b:
                nodes += 1

        return nodes

    def shoot(self, E, parity):
        E = mp.mpf(E)
        sol = self.solve_wavefunction(E, parity)

        if isinstance(sol, Exception):
            return ShootResult(
                energy=E,
                parity=parity,
                nodes_halfline=-1,
                psi_rmax=mp.nan,
                success=False,
                message=str(sol),
            )

        try:
            nodes = self.count_nodes_halfline(sol, parity)
            psi_rmax = sol(self.rmax)[0]
            return ShootResult(
                energy=E,
                parity=parity,
                nodes_halfline=nodes,
                psi_rmax=psi_rmax,
                success=True,
                message="ok",
            )
        except Exception as exc:
            return ShootResult(
                energy=E,
                parity=parity,
                nodes_halfline=-1,
                psi_rmax=mp.nan,
                success=False,
                message=str(exc),
            )

    @staticmethod
    def level_parity(level):
        return "even" if level % 2 == 0 else "odd"

    @staticmethod
    def target_halfline_nodes(level):
        return level // 2

    def classify_energy(self, E, parity):
        res = self.shoot(E, parity)
        if not res.success:
            raise RuntimeError(f"Shooting failed at E={E}: {res.message}")
        return res.nodes_halfline

    def find_bracket_for_level(self, level, E_min=None, E_max=None, n_scan=400):
        """
        Find [a,b] such that:
            nodes(a) <= target_nodes
            nodes(b) >  target_nodes
        """
        parity = self.level_parity(level)
        target_nodes = self.target_halfline_nodes(level)

        if E_min is None:
            E_min = mp.mpf(0)
        else:
            E_min = mp.mpf(E_min)

        if E_max is None:
            E_max = mp.mpf(max(10.0, 2.5 * (level + 1)))
        else:
            E_max = mp.mpf(E_max)

        step = (E_max - E_min) / n_scan

        prev_E = None
        prev_nodes = None

        for i in range(n_scan + 1):
            E = E_min + i * step
            try:
                nodes = self.classify_energy(E, parity)
            except RuntimeError:
                continue

            if prev_E is not None:
                if prev_nodes <= target_nodes and nodes > target_nodes:
                    return prev_E, E

            prev_E = E
            prev_nodes = nodes

        raise RuntimeError(
            f"Could not bracket level n={level}. "
            f"Try increasing E_max, rmax, or n_scan."
        )

    def bisect_energy(self, level, E_left, E_right, rel_tol=None, max_iter=200):
        """
        Node-count bisection.
        """
        parity = self.level_parity(level)
        target_nodes = self.target_halfline_nodes(level)

        E_left = mp.mpf(E_left)
        E_right = mp.mpf(E_right)

        if rel_tol is None:
            rel_tol = mp.mpf(10) ** (-(mp.mp.dps - 10))
        else:
            rel_tol = mp.mpf(rel_tol)

        left_nodes = self.classify_energy(E_left, parity)
        right_nodes = self.classify_energy(E_right, parity)

        if not (left_nodes <= target_nodes and right_nodes > target_nodes):
            raise RuntimeError(
                "Initial bracket does not straddle the node-count threshold."
            )

        a, b = E_left, E_right

        for _ in range(max_iter):
            c = (a + b) / 2
            mid_nodes = self.classify_energy(c, parity)

            if abs(b - a) < rel_tol * max(mp.mpf(1), abs(c)):
                return c

            if mid_nodes <= target_nodes:
                a = c
            else:
                b = c

        return (a + b) / 2

    def energy_level(
        self,
        level,
        E_min=None,
        E_max=None,
        n_scan=400,
        rel_tol=None,
        max_iter=200,
        verbose=False,
    ):
        a, b = self.find_bracket_for_level(level, E_min=E_min, E_max=E_max, n_scan=n_scan)
        if verbose:
            print(f"Level n={level}: bracket = [{a}, {b}]")
        return self.bisect_energy(level, a, b, rel_tol=rel_tol, max_iter=max_iter)

    def spectrum(
        self,
        n_levels,
        E_min=None,
        E_max=None,
        n_scan=400,
        rel_tol=None,
        max_iter=200,
        verbose=False,
    ):
        vals = []
        for n in range(n_levels):
            En = self.energy_level(
                level=n,
                E_min=E_min,
                E_max=E_max,
                n_scan=n_scan,
                rel_tol=rel_tol,
                max_iter=max_iter,
                verbose=verbose,
            )
            vals.append(En)
            if verbose:
                print(f"E_{n} = {En}")
        return vals

In [20]:
solver_phi4 = AnharmonicOscillator1DMP(
    lam4=1,
    lam6=0,
    rmax=5,
    dps=50,
    n_count_grid=3000,
)

E1 = solver_phi4.energy_level(1, E_min="1.6", E_max="1.7", n_scan=200, verbose=True)
E2 = solver_phi4.energy_level(2, E_min="2.8", E_max="2.9", n_scan=200, verbose=True)
E3 = solver_phi4.energy_level(3, E_min="4.0", E_max="4.1", n_scan=200, verbose=True)

print(f"E1 = {E1}")
print(f"E2 = {E2}")
print(f"E3 = {E3}")

Level n=1: bracket = [1.631, 1.6315]
Level n=2: bracket = [2.822, 2.8225]
Level n=3: bracket = [4.086, 4.0865]
E1 = 1.631300532768307447535797641299407409666056688344
E2 = 2.8221895629546794107586487866983585728937188414279
E3 = 4.0860289543126751347062043012369318729517761811902


In [16]:
print(E0)
print(E1)

0.53
1.6


In [4]:
solver_phi4 = AnharmonicOscillator1DMP(
    lam4=32,
    lam6=0,
    rmax=5,
    dps=50,
    n_count_grid=3000,
)

E0 = solver_phi4.energy_level(0, E_min="0.8", E_max="0.9", n_scan=200, verbose=True)

Level n=0: bracket = [0.8595, 0.86]


In [5]:
print(E0)

0.8597426904455090193559617737854627804109812323184
